# Click-through-rate prediction
Objective: predict CTR based on TaoBao's datasets.<br>
Motivation: In online advertising systems, CTR prediction is the backbone of auction mechanism and used to decide which advertisement should be displayed to users, how much the advertisers should be charged, etc.<br>
[Link to datasets](https://tianchi.aliyun.com/dataset/56?spm=a2c22.12282016.0.0.27932e70F6FmRS)<br>
This notebook contains the following parts:
> 1. Exploratory data analysis
> 2. Preprocessing
> 3. Modeling<br>

In [ ]:
# Uncomment and run if you dont have the below libraries
%pip install -U pandas numpy matplotlib scikit-learn imbalanced-learn xgboost torch gdown

In [ ]:
# Load libraries
import tarfile
import io
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import gdown

In [ ]:
#Download datasets from google drive. This worked (fast) for me as I was working on remote devices.
file_configs = [
    ("user_profile.csv.tar.gz", "1R5jHkFLsJYSi3yoV3zCI9Buj_rjGLhqC"),
    ("ad_feature.csv.tar.gz", "1OvZgCDBE8usTfWXBVzuqB_cGke6FRPB0"),
    ("raw_sample.csv.tar.gz", "1-a9Nvt1jGunGjJc394MxSxUiDmzfGWur"),
]

os.makedirs("data", exist_ok=True)

for filename, file_id in file_configs:
    if file_id:
        url = f"https://drive.google.com/uc?id={file_id}"
        output = f"data/{filename}"
        gdown.download(url, output, quiet=False)
        print(f"Downloaded {filename}")
    else:
        print(f"Skip {filename}")

print("Done")

In [ ]:
#Unpack the data
import os
import tarfile
import pandas as pd

data_paths = ["data/user_profile.csv.tar.gz",
              "data/ad_feature.csv.tar.gz",
              "data/raw_sample.csv.tar.gz",
              ]
datasets = {}
for path in data_paths:
    with tarfile.open(path, "r:gz") as tar:
        csv_files = [name for name in tar.getnames() if name.endswith('.csv')]
        csv_path = csv_files[0]
        dataset_name = os.path.basename(path).replace('.csv.tar.gz', '').replace('.tar.gz', '')

        # Extract
        tar.extract(csv_path, path="data/")
        csv_file_path = f"data/{csv_path}"

        datasets[dataset_name] = pd.read_csv(csv_file_path)
        print(f"Load {dataset_name} done ({len(datasets[dataset_name])} rows)")

## 1. Exploratory data analysis

In [ ]:
# First look at datasets
user_profile = datasets['user_profile']
ad_feature = datasets['ad_feature']
raw_sample = datasets['raw_sample']

display(user_profile.info())
print("------------------------------------------------------------------------")
display(ad_feature.info())
print("------------------------------------------------------------------------")
display(raw_sample.info())

In [ ]:
# Some distributions of data in ad_feature table
unique_brand = ad_feature["brand"].nunique()
unique_advertiser = ad_feature["customer"].nunique()
max_price = ad_feature["price"].max()
min_price = ad_feature["price"].min()
p75 = ad_feature["price"].quantile(0.75)
p25 = ad_feature["price"].quantile(0.25)

# Print out values
print(f"Number of unique brands: {unique_brand}")
print(f"Number of unique advertisers: {unique_advertiser}")
print(f"Max: {max_price}")
print(f"Min: {min_price}")
print(f"Price 75th percentile: {p75}")
print(f"Price 25th percentile: {p25}")
# => The price is very stretched across brands with the range from 0.01 (probably in RMB) to 99999999.
# Nonetheless, values like 9999999 are probably outliers that's worth removed (No item on an e-commerce web will reach this kind of price).
# We will remove the 1% highest-priced item for later analysis.
p99 = ad_feature["price"].quantile(0.99)
ad_feature[ad_feature["price"] < p99]["price"].hist(bins=50)
plt.xlabel("price")
plt.title("price distribution")
plt.show()

In [ ]:
# Distributions of data in user_profile table
# Rename column 'new_user_class_level '
user_profile.rename(
    columns={'new_user_class_level ': "new_user_class_level"}, inplace=True)

user_cols = user_profile.columns[3:]  # Ignore id columns

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(8, 6))
axes = axes.flatten()

for i, col in enumerate(user_cols):
    ax = axes[i]
    category_count = user_profile[col].value_counts()
    category_count.plot(kind="bar", ax=ax)

    ax.set_title(col)
    ax.set_ylabel("Count")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

# 2. Preprocess
We will use raw_samples as our main table, then add features from the other tables. First of all, we will join the tables, then split data into train and test sets based on time_stap to avoid data leakage. Specifically:
- Train: From 2017-05-05 to 2017-05-12
- Test: On 2017-05-13

In [ ]:
# Process raw_sample table
raw_sample["time_stamp"] = pd.to_datetime(raw_sample["time_stamp"], unit="s")

display(raw_sample["time_stamp"].max())  # '2017-05-13 15:59:46'
display(raw_sample["time_stamp"].min())  # '2017-05-05 16:00:00'

# Add hour and day features
raw_sample["day"] = raw_sample["time_stamp"].dt.day
raw_sample["hour"] = raw_sample["time_stamp"].dt.hour

display(raw_sample.head(3))

In [ ]:
# Create a table includes all features
from_raw = raw_sample[["user", "adgroup_id", "clk", "pid", "hour", "day"]]
from_ad = ad_feature[["adgroup_id", "customer", "brand", "price"]]
from_user = user_profile[["userid", "final_gender_code", "age_level", "cms_segid", "cms_group_id",
                          "pvalue_level", "shopping_level", "occupation", "new_user_class_level"]]

merge1 = pd.merge(from_raw, from_ad, left_on="adgroup_id",
                  right_on="adgroup_id", how="left")
final_table = pd.merge(merge1, from_user, left_on="user",
                       right_on="userid", how="inner")
final_table = final_table.drop("userid", axis=1)

In [ ]:
#Check outliers in price column
plt.boxplot(final_table["price"])
plt.show()

In [ ]:
# Remove price values that is at p99 and higher in price column

p99 = final_table["price"].quantile(0.99)
final_table = final_table[final_table["price"] < p99]

# Standardize "price" column. As the column is strongly right-skewed, I will take the log of it first, then standardizing it afterwards.
final_table["price"] = np.log1p(final_table["price"])

In [ ]:
# Brand, pvalue_level, new_user_class_level have null values. pid has string data.
final_table.info()
final_table.isnull().sum()

In [ ]:
# Process null values in final table.

# As the missing value is simply unknown, it might still contains information, we will fill with 0, then perform onehot encoding later.
columns_with_na = ["brand", "pvalue_level", "new_user_class_level"]
final_table[columns_with_na] = final_table[columns_with_na].fillna(0)

# Also, we add 2 corresponding binary  columns to indicate which row is unknown.
final_table["brand_ismissing"] = (
    final_table["brand"].isna()).astype(int)
final_table["pvalue_level_ismissing"] = (
    final_table["pvalue_level"].isna()).astype(int)
final_table["new_user_class_level_ismissing"] = (
    final_table["new_user_class_level"].isna()).astype(int)

final_table

In [ ]:
# Split data into train, test sets (time-based)
boo_mask = final_table["day"].between(5, 12)
train_set = final_table[boo_mask]
test_set = final_table[~boo_mask]

X_train = train_set.iloc[:, train_set.columns != 'clk']
y_train = train_set.iloc[:, train_set.columns == 'clk']
X_test = test_set.iloc[:, test_set.columns != 'clk']
y_test = test_set.iloc[:, test_set.columns == 'clk']
# Double check
print(f"Total # rows: {len(final_table)}")
print(f"Train set # rows: {len(train_set)}")
print(f"Test set # rows: {len(test_set)}")

In [ ]:
# Encode nominal category data using target encoding
nominal_cols = ['user', 'adgroup_id', 'pid', 'customer', 'brand',
                'final_gender_code', 'age_level',
                'cms_segid', 'cms_group_id', 'occupation'
                ]
X_train[nominal_cols] = X_train[nominal_cols].astype(float)
X_test[nominal_cols] = X_test[nominal_cols].astype(float)

target_encoder = TargetEncoder(target_type="binary",  smooth="auto", cv=5)
X_train.loc[:, nominal_cols] = target_encoder.fit_transform(
    X_train[nominal_cols], y_train["clk"])
X_test.loc[:, nominal_cols] = target_encoder.transform(X_test[nominal_cols])

In [ ]:
# Onehot encoding for ordinal columns with missing values
ordinal_cols_missing = ["pvalue_level", "new_user_class_level"]
oh_encoder = OneHotEncoder(sparse_output=False,
                           drop='first',
                           # Return dense array, drop one category, set handl_unknown="ignore" to handle 
                           # new category in test set (if any)
                           handle_unknown='ignore')

ohe_train = oh_encoder.fit_transform(X_train[ordinal_cols_missing])
ohe_test = oh_encoder.transform(X_test[ordinal_cols_missing])

feature_names = oh_encoder.get_feature_names_out(ordinal_cols_missing)
X_train[feature_names] = ohe_train
X_test[feature_names] = ohe_test

X_train.drop(columns=ordinal_cols_missing, inplace=True)
X_test.drop(columns=ordinal_cols_missing, inplace=True)

In [ ]:
#Standardize price column
X_train["price"] = X_train["price"].astype(float)
X_test["price"] = X_test["price"].astype(float)
scaler = StandardScaler()
X_train[["price"]] = scaler.fit_transform(X_train[["price"]])
X_test[["price"]] = scaler.transform(X_test[["price"]])

In [ ]:
# Final features
X_train.info()

## 3. Modeling

As most advertising bidding/auction systems require individual click probability given a new record, I experimented with several models to compare performance. 
Note: there are some other well-known models for the problem that I haven't tried, due to limited computing resources including KNN, random forest, transformer-based neural network, also the randomized search cross-validation for optimal deicision tree and random forest.

## 3.1. Logistic regression

A simple logistic regression can yield an AUC at 0.62 (baseline 0.62) and log-loss 0.2.

In [ ]:
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train["clk"])
lr_y_pred = lr.predict_proba(X_test)
lr_auc = roc_auc_score(y_test, lr_y_pred[:, 1])
lr_logloss = log_loss(y_test, lr_y_pred)

print(f"ROC AUC: {lr_auc:.2f}")
print(f"Log-loss: {lr_logloss:.2f}")

In [ ]:
#Extract the model coefficients to identify which features have the greatest explanatory power on the target variable.
coefficients = lr.coef_[0]
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': coefficients
}).sort_values('Coefficient', ascending=False)
feature_importance

## 3.2. Decision Tree

The decision tree outperforms logistic regression and reach an AUC at 0.63, but with higher log-loss at 0.67

In [ ]:
#Final training
dt = DecisionTreeClassifier(
    max_depth=6,           
    min_samples_leaf=50,   
    class_weight='balanced',  
    random_state=42
)
dt.fit(X_train, y_train["clk"])

dt_y_pred = dt.predict_proba(X_test)
dt_auc = roc_auc_score(y_test, dt_y_pred[:, 1])
dt_logloss = log_loss(y_test, dt_y_pred)
print(f"ROC AUC: {dt_auc:.2f}")
print(f"Log-loss: {dt_logloss:.2f}")

In [ ]:
#Similar to logistic regression, get feature importances from fitted decision tree
importances = dt.feature_importances_ 
feature_names = X_train.columns 

feat_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

feat_imp

## 3.3. XGBoost

XGBoost yields poor performance with an AUC of 0.54, but with lower log-loss at 0.20

In [ ]:
xgb = XGBClassifier(
    random_state=42,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    tree_method='hist'
)

xgb.fit(X_train, y_train["clk"])

xgb_y_pred = xgb.predict_proba(X_test)
xgb_auc = roc_auc_score(y_test, xgb_y_pred[:, 1])
xgb_logloss = log_loss(y_test, xgb_y_pred)

print(f"XGBoost AUC:      {xgb_auc:.2f}")
print(f"XGBoost Log-loss: {xgb_logloss:.2f}")

## 3.4. Multiple-layer perceptron

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.model(x)

def train_mlp(X_train, y_train, epochs=10):
    X_tensor = torch.tensor(X_train.values, dtype=torch.float32)
    y_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1)

    dataset = TensorDataset(X_tensor, y_tensor)
    train_loader = DataLoader(dataset, batch_size=1024, shuffle=True, num_workers=4, pin_memory=True)

    model = MLP(input_dim=X_train.shape[1])
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=3e-3)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for X_batch, y_batch in train_loader:
            logits = model(X_batch).squeeze(1)
            loss = loss_fn(logits, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {avg_loss:.4f}")

    return model

In [ ]:
#Train
model = train_mlp(X_train, y_train)

In [ ]:
#Eval
model.eval()
with torch.no_grad():
    X_tensor = torch.tensor(X_test.values, dtype=torch.float32)
    logits = model(X_tensor).squeeze(1)
    probs = torch.sigmoid(logits).numpy()
nn_auc = roc_auc_score(y_test, probs)
nn_logloss = log_loss(y_test, probs)
print(f"MLP AUC:      {nn_auc:.2f}")
print(f"MLP Log-loss: {nn_logloss:.2f}")

References:
- What metrics to use for classification task: https://developers.google.com/machine-learning/crash-course/classification/accuracy-precision-recall
- How to handle imbalanced dataset: https://developers.google.com/machine-learning/crash-course/overfitting/imbalanced-datasets